---
## Cell 0 — Master Configuration

In [7]:
import os, torch
from pathlib import Path

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE   = 16
GRAD_ACCUM   = 2
N_WORKERS    = 2
N_FOLDS      = 3
MAX_EPOCHS   = 25
PATIENCE     = 6
LR           = 3e-4
WEIGHT_DECAY = 1e-4
SMOKE_N      = 50

# ── EEG signal config (must match V2 exactly) ─────────────────────────────────
SFREQ       = 200
WIN_SEC     = 50
LABEL_SEC   = 10
WIN_SAMP    = SFREQ * WIN_SEC    # 10000
LABEL_SAMP  = SFREQ * LABEL_SEC # 2000
LABEL_START = (WIN_SAMP - LABEL_SAMP) // 2
LABEL_END   = LABEL_START + LABEL_SAMP
MIN_VOTES   = 8

LL = [('Fp1','F7'),('F7','T3'),('T3','T5'),('T5','O1')]
LP = [('Fp1','F3'),('F3','C3'),('C3','P3'),('P3','O1')]
RP = [('Fp2','F4'),('F4','C4'),('C4','P4'),('P4','O2')]
RL = [('Fp2','F8'),('F8','T4'),('T4','T6'),('T6','O2')]
BIPOLAR_PAIRS = LL + LP + RP + RL
N_CH          = len(BIPOLAR_PAIRS)   # 16
CLASSES       = ['seizure','lpd','gpd','lrda','grda','other']
N_CLASSES     = 6

# ── Spectrogram config ────────────────────────────────────────────────────────
SPEC_H      = 224     # height (frequency bins after resize)
SPEC_W      = 224     # width  (time bins after resize)
N_FFT       = 256     # FFT window
HOP_LENGTH  = 32      # hop between windows  (≈ 160 ms overlap @ 200 Hz)
N_MELS      = 64      # Mel filter banks
FMIN        = 0.5     # Hz
FMAX        = 50.0    # Hz (< Nyquist 100 Hz)
SPEC_CACHE  = True    # cache spectrograms to disk on first run
MAX_CACHE_N = 17_000  # cap to stay within Kaggle RAM

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE            = Path('/kaggle/input/competitions/hms-harmful-brain-activity-classification')
TRAIN_CSV       = BASE / 'train.csv'
TRAIN_EEG       = BASE / 'train_eegs'
OUT_DIR         = Path('/kaggle/working')
MODEL_DIR       = OUT_DIR / 'models'
SPEC_CACHE_DIR  = OUT_DIR / 'spec_cache'
MODEL_DIR.mkdir(exist_ok=True)
SPEC_CACHE_DIR.mkdir(exist_ok=True)

print(f'Device       : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU          : {props.name}')
    print(f'VRAM         : {props.total_memory/1e9:.1f} GB')
print(f'Spectrogram  : {N_CH} channels × {SPEC_H}×{SPEC_W} px')
print(f'Batch        : {BATCH_SIZE} × accum {GRAD_ACCUM} = eff. {BATCH_SIZE*GRAD_ACCUM}')
print('Config ready ✓')

Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
Spectrogram  : 16 channels × 224×224 px
Batch        : 16 × accum 2 = eff. 32
Config ready ✓


---
## Cell 1 — Imports & Installation

In [8]:
# Install librosa for Mel spectrogram (quiet install)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'librosa', '-q'], check=False)

import gc, warnings, random, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
from typing import Optional, Tuple, List, Dict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import torchvision.models as tv_models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                              roc_curve, auc as sklearn_auc)
from scipy import signal as scipy_signal
import librosa
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print('Imports done ✓')

Imports done ✓


---
## Cell 2 — Data Audit
Identical patient-level split strategy as V2 — no leakage.

In [9]:
print('Loading train.csv ...')
df_raw = pd.read_csv(TRAIN_CSV)
print(f'  Rows × cols : {df_raw.shape}')

# Drop null patient_id (same guard as V2)
df_raw = df_raw.dropna(subset=['patient_id']).reset_index(drop=True)

VOTE_COLS = [f'{c}_vote' for c in CLASSES]
PROB_COLS = [f'{c}_prob' for c in CLASSES]
assert all(c in df_raw.columns for c in VOTE_COLS), 'Missing vote columns'

df_raw['total_votes'] = df_raw[VOTE_COLS].sum(axis=1)
for c in CLASSES:
    df_raw[f'{c}_prob'] = df_raw[f'{c}_vote'] / df_raw['total_votes']

# Aggregate to one row per eeg_id (same as V2)
eeg_meta = df_raw.groupby('eeg_id').agg(
    patient_id       = ('patient_id', 'first'),
    expert_consensus = ('expert_consensus', lambda x: x.mode()[0]),
    **{f'{c}_vote': (f'{c}_vote', 'sum') for c in CLASSES}
).reset_index()
eeg_meta['total_votes'] = eeg_meta[[f'{c}_vote' for c in CLASSES]].sum(axis=1)
for c in CLASSES:
    eeg_meta[f'{c}_prob'] = eeg_meta[f'{c}_vote'] / eeg_meta['total_votes']

# High-confidence subset (MIN_VOTES)
eeg_high = eeg_meta[eeg_meta['total_votes'] >= MIN_VOTES].reset_index(drop=True)
print(f'  Total EEG IDs  : {len(eeg_meta):,}')
print(f'  High-conf IDs  : {len(eeg_high):,} (≥{MIN_VOTES} votes)')
print(f'  Unique patients: {eeg_high["patient_id"].nunique():,}')

# Class balance
print('\nClass distribution (high-conf):')
for c in CLASSES:
    n = (eeg_high['expert_consensus'] == c).sum()
    pct = 100*n/len(eeg_high)
    print(f'  {c:<10}: {n:5,}  ({pct:.1f}%)')

# Quick EEG file sanity
n_found = sum(1 for eid in eeg_high['eeg_id'].values[:20]
               if (TRAIN_EEG / f'{eid}.parquet').exists())
print(f'\nEEG file check (first 20): {n_found}/20 found')
print('Data audit done ✓')

Loading train.csv ...
  Rows × cols : (106800, 15)
  Total EEG IDs  : 17,089
  High-conf IDs  : 10,897 (≥8 votes)
  Unique patients: 1,888

Class distribution (high-conf):
  seizure   :     0  (0.0%)
  lpd       :     0  (0.0%)
  gpd       :     0  (0.0%)
  lrda      :     0  (0.0%)
  grda      :     0  (0.0%)
  other     :     0  (0.0%)

EEG file check (first 20): 20/20 found
Data audit done ✓


---
## Cell 3 — Spectrogram Generator

We generate **Mel spectrograms** per bipolar channel:
1. Compute STFT-based Mel filterbank (`librosa.feature.melspectrogram`)
2. Convert to dB scale (`librosa.power_to_db`)
3. Normalise to [0, 1] per channel
4. Resize to `SPEC_H × SPEC_W` with bilinear interpolation

Mel scale is preferred over linear STFT because:
- It matches the perceptual frequency resolution of EEG diagnostics (emphasises low-freq patterns like delta/theta)
- Seizures and LRDA often manifest as broadband or low-frequency bursts visible in Mel representation
- Consistent with published EEG-image deep learning literature

In [10]:
def bandpass_filter(sig: np.ndarray, lowcut=0.5, highcut=50.0, fs=200, order=4) -> np.ndarray:
    """Butterworth bandpass filter."""
    nyq = fs / 2.0
    b, a = scipy_signal.butter(
        order,
        [lowcut / nyq, highcut / nyq],
        btype='band'
    )

    try:
        sig = scipy_signal.filtfilt(b, a, sig, axis=-1)
    except Exception:
        sig = np.nan_to_num(
            sig,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

    return sig


def eeg_to_bipolar(df_eeg: pd.DataFrame) -> np.ndarray:
    """
    Convert raw EEG parquet columns to 16-channel bipolar montage.
    Returns shape (16, LABEL_SAMP)
    """

    signals = []

    for (a, b) in BIPOLAR_PAIRS:

        ch_a = (
            df_eeg[a].values
            if a in df_eeg.columns
            else np.zeros(len(df_eeg))
        )

        ch_b = (
            df_eeg[b].values
            if b in df_eeg.columns
            else np.zeros(len(df_eeg))
        )

        diff = ch_a - ch_b

        
        diff = np.nan_to_num(
            diff,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        diff = np.clip(diff, -1024.0, 1024.0)

        signals.append(diff)

    raw = np.stack(signals, axis=0)

    # center crop
    T = raw.shape[1]
    start = max(0, (T - LABEL_SAMP) // 2)

    raw = raw[:, start:start + LABEL_SAMP]

    if raw.shape[1] < LABEL_SAMP:
        pad = LABEL_SAMP - raw.shape[1]
        raw = np.pad(raw, ((0, 0), (0, pad)))
    else:
        raw = raw[:, :LABEL_SAMP]

    
    raw = np.nan_to_num(
        raw,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return raw


def compute_mel_spectrogram(
    waveform: np.ndarray,
    sr: int = SFREQ,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
    n_mels: int = N_MELS,
    fmin: float = FMIN,
    fmax: float = FMAX,
) -> np.ndarray:
    """
    Returns normalized log-Mel spectrogram.
    """

    
    waveform = np.asarray(
        waveform,
        dtype=np.float32
    )

    waveform = np.nan_to_num(
        waveform,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    # avoid all-zero channels
    if np.std(waveform) < 1e-8:
        waveform = waveform + np.random.normal(
            0,
            1e-6,
            waveform.shape
        ).astype(np.float32)

    # bandpass
    waveform = bandpass_filter(
        waveform,
        lowcut=fmin,
        highcut=fmax,
        fs=sr
    )

    
    waveform = np.nan_to_num(
        waveform,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    mel = librosa.feature.melspectrogram(
        y=waveform,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        power=2.0
    )

    mel = np.nan_to_num(
        mel,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    mel_db = np.nan_to_num(
        mel_db,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    mel_db -= mel_db.min()

    denom = mel_db.max()

    if denom > 1e-6:
        mel_db /= denom

    return mel_db.astype(np.float32)


def build_spectrogram_tensor(
    bipolar_signals: np.ndarray,
    spec_h: int = SPEC_H,
    spec_w: int = SPEC_W,
) -> torch.Tensor:
    """
    Convert 16-channel EEG into
    (16, spec_h, spec_w)
    """

    ch_specs = []

    for ch in range(bipolar_signals.shape[0]):
        channel = np.nan_to_num(
            bipolar_signals[ch],
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        spec = compute_mel_spectrogram(channel)

        ch_specs.append(spec)

    arr = np.stack(ch_specs, axis=0)

    arr = np.nan_to_num(
        arr,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    t = torch.from_numpy(arr).unsqueeze(0)

    t = F.interpolate(
        t,
        size=(spec_h, spec_w),
        mode='bilinear',
        align_corners=False
    )

    return t.squeeze(0)


print("Spectrogram generator defined ✓")
print(
    f"Mel params: n_fft={N_FFT}, "
    f"hop={HOP_LENGTH}, "
    f"n_mels={N_MELS}, "
    f"[{FMIN}, {FMAX}] Hz"
)
print(f"Output shape per EEG: (16, {SPEC_H}, {SPEC_W})")

Spectrogram generator defined ✓
Mel params: n_fft=256, hop=32, n_mels=64, [0.5, 50.0] Hz
Output shape per EEG: (16, 224, 224)


---
## Cell 4 — HMSSpectrogramDataset + Augmentations

Augmentations applied **during training only**:
- **Time masking** — randomly zero out T consecutive time columns
- **Frequency masking** — randomly zero out F consecutive frequency rows
- **Random gain** — multiply all channels by a random scalar
- **Temporal shift** — circular-shift spectrogram by a small random amount
- **Gaussian noise** — add low-amplitude Gaussian noise

In [11]:
# ─── Augmentations ────────────────────────────────────────────────────────────

def time_mask(spec: torch.Tensor, max_t: int = 30) -> torch.Tensor:
    _, H, W = spec.shape
    t = random.randint(1, max_t)
    t0 = random.randint(0, max(0, W - t))
    spec = spec.clone()
    spec[:, :, t0:t0+t] = 0.0
    return spec

def freq_mask(spec: torch.Tensor, max_f: int = 20) -> torch.Tensor:
    _, H, W = spec.shape
    f = random.randint(1, max_f)
    f0 = random.randint(0, max(0, H - f))
    spec = spec.clone()
    spec[:, f0:f0+f, :] = 0.0
    return spec

def random_gain(spec: torch.Tensor, low: float = 0.8, high: float = 1.2) -> torch.Tensor:
    g = random.uniform(low, high)
    return (spec * g).clamp(0.0, 1.0)

def temporal_shift(spec: torch.Tensor, max_shift: int = 16) -> torch.Tensor:
    shift = random.randint(-max_shift, max_shift)
    return torch.roll(spec, shifts=shift, dims=2)

def gaussian_noise(spec: torch.Tensor, std: float = 0.02) -> torch.Tensor:
    return (spec + torch.randn_like(spec) * std).clamp(0.0, 1.0)

def augment_spectrogram(spec: torch.Tensor) -> torch.Tensor:
    if random.random() < 0.5:
        spec = time_mask(spec)

    if random.random() < 0.5:
        spec = freq_mask(spec)

    if random.random() < 0.4:
        spec = random_gain(spec)

    if random.random() < 0.3:
        spec = temporal_shift(spec)

    if random.random() < 0.3:
        spec = gaussian_noise(spec)

    return spec


# ─── Dataset ──────────────────────────────────────────────────────────────────

class HMSSpectrogramDataset(Dataset):

    def __init__(
        self,
        meta_df: pd.DataFrame,
        eeg_dir: Path,
        train: bool = True,
        cache: bool = SPEC_CACHE,
        cache_dir: Path = SPEC_CACHE_DIR,
    ):
        self.meta = meta_df.reset_index(drop=True)
        self.eeg_dir = eeg_dir
        self.train = train
        self.cache = cache
        self.cache_dir = cache_dir
        self._memory = {}

    def __len__(self):
        return len(self.meta)

    def _load_spec(self, eeg_id: int) -> torch.Tensor:

        # ---------- Memory cache ----------
        if eeg_id in self._memory:
            return self._memory[eeg_id]

        cache_path = self.cache_dir / f"{eeg_id}.pt"

        # ---------- Disk cache ----------
        if self.cache and cache_path.exists():

            try:
                spec = torch.load(
                    cache_path,
                    map_location="cpu"
                )

                if len(self._memory) < MAX_CACHE_N:
                    self._memory[eeg_id] = spec

                return spec

            except Exception:

                # corrupted cache file
                try:
                    cache_path.unlink()
                except:
                    pass

        # ---------- Compute ----------
        parquet = self.eeg_dir / f"{eeg_id}.parquet"

        if parquet.exists():

            try:
                df_eeg = pd.read_parquet(parquet)

            except Exception:

                return torch.zeros(
                    N_CH,
                    SPEC_H,
                    SPEC_W,
                    dtype=torch.float32
                )

        else:

            return torch.zeros(
                N_CH,
                SPEC_H,
                SPEC_W,
                dtype=torch.float32
            )

        bipolar = eeg_to_bipolar(df_eeg)

        spec = build_spectrogram_tensor(bipolar)

        spec = torch.nan_to_num(
            spec,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        # ---------- Safe cache write ----------
        if self.cache:

            try:
                torch.save(spec, cache_path)

            except Exception:
                pass

        # ---------- Memory cache ----------
        if len(self._memory) < MAX_CACHE_N:
            self._memory[eeg_id] = spec

        return spec

    def __getitem__(self, idx: int):

        row = self.meta.iloc[idx]

        eeg_id = int(row["eeg_id"])
        pid = int(row["patient_id"])

        labels = torch.tensor(
            [float(row[f"{c}_prob"]) for c in CLASSES],
            dtype=torch.float32
        )

        spec = self._load_spec(eeg_id).float()

        spec = torch.nan_to_num(
            spec,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        if self.train:
            spec = augment_spectrogram(spec)

        return spec, labels, pid


print("HMSSpectrogramDataset defined ✓")
print("Safe cache enabled ✓")
print("Corrupted cache recovery enabled ✓")
print("Augmentations enabled ✓")

HMSSpectrogramDataset defined ✓
Safe cache enabled ✓
Corrupted cache recovery enabled ✓
Augmentations enabled ✓


---
## Cell 5 — Spectrogram Model: EfficientNet-B0 (16-channel)

**Design decisions:**
- EfficientNet-B0 pretrained on ImageNet
- First Conv2d replaced: `(32, 3, 3, 3)` → `(32, 16, 3, 3)` to accept 16 EEG channels
  - Pretrained RGB weights are mean-replicated across 16 input channels to warm-start
- Classifier head replaced: `(1280 → 6)` with dropout
- BatchNorm layers frozen for first `FREEZE_BN_EPOCHS` to stabilise training with small batches

In [12]:
FREEZE_BN_EPOCHS = 3


class SpectrogramEfficientNet(nn.Module):
    """
    EfficientNet-B0 adapted for 16-channel EEG Mel spectrograms.

    Input  : (B, 16, 224, 224)
    Output : (B, 6) logits
    """

    def __init__(self, n_classes: int = N_CLASSES, dropout: float = 0.4):
        super().__init__()

        # Load pretrained backbone
        backbone = tv_models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        # ── Adapt first conv: 3 → 16 input channels ──────────────────────────
        old_conv  = backbone.features[0][0]  # Conv2d(3, 32, 3, stride=2)
        new_conv  = nn.Conv2d(
            in_channels=N_CH,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        # Warm-start: replicate RGB weights across 16 channels and rescale
        with torch.no_grad():
            rgb_w = old_conv.weight.data       # (32, 3, 3, 3)
            rep   = rgb_w.repeat(1, 6, 1, 1)  # (32, 18, 3, 3) — repeat 6× then slice
            new_conv.weight.data = rep[:, :N_CH, :, :] * (3.0 / N_CH)
        backbone.features[0][0] = new_conv

        # ── Replace classifier head ───────────────────────────────────────────
        in_features = backbone.classifier[1].in_features  # 1280
        backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.SiLU(),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, n_classes),
        )

        self.backbone = backbone

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 16, H, W) → (B, 6) logits"""
        return self.backbone(x)


def build_model() -> SpectrogramEfficientNet:
    return SpectrogramEfficientNet().to(DEVICE)


# Quick shape check
_m = build_model()
_x = torch.randn(2, N_CH, SPEC_H, SPEC_W).to(DEVICE)
with torch.no_grad():
    _o = _m(_x)
print(f'Model output shape : {_o.shape}   (expected: [2, 6])')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable params   : {n_params/1e6:.2f} M')
del _m, _x, _o
torch.cuda.empty_cache()
print('SpectrogramEfficientNet defined ✓')

Model output shape : torch.Size([2, 6])   (expected: [2, 6])
Trainable params   : 4.34 M
SpectrogramEfficientNet defined ✓


---
## Cell 6 — Loss Function & Trainer

In [19]:
# ─── KL Divergence Loss (soft labels) ────────────────────────────────────────

def kl_loss(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    log_pred = F.log_softmax(logits, dim=-1)
    return F.kl_div(
        log_pred,
        targets.clamp(min=eps),
        reduction='batchmean'
    )


# ─── WeightedRandomSampler ───────────────────────────────────────────────────

def build_sampler(meta_df: pd.DataFrame) -> WeightedRandomSampler:

    cls_counts = meta_df['expert_consensus'].value_counts().to_dict()

    weights = meta_df['expert_consensus'].map(
        lambda c: 1.0 / cls_counts.get(c, 1)
    ).values

    return WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.float64),
        num_samples=len(meta_df),
        replacement=True,
    )


# ─── Trainer ─────────────────────────────────────────────────────────────────

class SpectrogramTrainer:

    def __init__(self, model: nn.Module, fold: int = 0):

        self.model = model
        self.fold = fold

        self.scaler = GradScaler(enabled=USE_AMP)

        self.opt = AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        self.sched = None

        self.best_kl = float('inf')
        self.best_ep = 0
        self.patience_count = 0

    # ─────────────────────────────────────────────────────────────

    def _run_epoch(
        self,
        loader: DataLoader,
        train: bool,
        epoch: int,
    ):

        self.model.train(train)

        total_loss = 0.0
        all_preds = []
        all_targets = []

        self.opt.zero_grad()

        for step, (specs, labels, _) in enumerate(
            tqdm(
                loader,
                desc='train' if train else 'val',
                leave=False
            )
        ):

            specs = specs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):

                logits = self.model(specs)

                loss = kl_loss(
                    logits,
                    labels
                )

                if train:
                    loss = loss / GRAD_ACCUM

            if train:

                self.scaler.scale(loss).backward()

                if (step + 1) % GRAD_ACCUM == 0:

                    self.scaler.unscale_(self.opt)

                    nn.utils.clip_grad_norm_(
                        self.model.parameters(),
                        1.0
                    )

                    self.scaler.step(self.opt)
                    self.scaler.update()

                    self.opt.zero_grad()

                    if self.sched is not None:
                        self.sched.step()

            total_loss += (
                loss.item()
                * (GRAD_ACCUM if train else 1)
            )

            probs = F.softmax(
                logits.detach(),
                dim=-1
            ).cpu().numpy()

            all_preds.append(probs)
            all_targets.append(labels.cpu().numpy())

        preds = np.vstack(all_preds)
        targets = np.vstack(all_targets)

        avg_kl = total_loss / len(loader)

        # ===== FIXED AUC =====

        try:

            aucs = []

            true_cls = targets.argmax(axis=1)

            for c in range(len(CLASSES)):

                y_true = (true_cls == c).astype(int)

                if y_true.sum() == 0:
                    continue

                if y_true.sum() == len(y_true):
                    continue

                aucs.append(
                    roc_auc_score(
                        y_true,
                        preds[:, c]
                    )
                )

            auc = float(np.mean(aucs)) if len(aucs) else 0.0

        except Exception as e:

            print("AUC ERROR:", e)
            auc = 0.0

        return avg_kl, auc

    # ─────────────────────────────────────────────────────────────

    def train(
        self,
        train_ds: HMSSpectrogramDataset,
        val_ds: HMSSpectrogramDataset,
        max_epochs: int = MAX_EPOCHS,
    ):

        print(f"Saving models to: {MODEL_DIR}")

        MODEL_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        sampler = build_sampler(
            train_ds.meta
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=True,
            drop_last=True,
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE * 2,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=True,
        )

        steps_per_epoch = len(train_loader)

        total_steps = (
            max_epochs
            * steps_per_epoch
        )

        self.sched = CosineAnnealingLR(
            self.opt,
            T_max=total_steps,
            eta_min=1e-6
        )

        history = {
            'train_kl': [],
            'val_kl': [],
            'val_auc': [],
        }

        for epoch in range(
            1,
            max_epochs + 1
        ):

            if epoch <= FREEZE_BN_EPOCHS:

                for m in self.model.modules():

                    if isinstance(
                        m,
                        nn.BatchNorm2d
                    ):
                        m.eval()

            tr_kl, _ = self._run_epoch(
                train_loader,
                train=True,
                epoch=epoch
            )

            val_kl, val_auc = self._run_epoch(
                val_loader,
                train=False,
                epoch=epoch
            )

            history['train_kl'].append(tr_kl)
            history['val_kl'].append(val_kl)
            history['val_auc'].append(val_auc)

            print(
                f'  Epoch {epoch:2d} | '
                f'tr_kl={tr_kl:.4f} | '
                f'val_kl={val_kl:.4f} | '
                f'val_auc={val_auc:.4f}'
            )

            # ===== SAVE BEST =====

            if val_kl < self.best_kl:

                self.best_kl = val_kl
                self.best_ep = epoch
                self.patience_count = 0

                ckpt_path = MODEL_DIR / f"spec_fold{self.fold}.pt"

                try:

                    torch.save(
                        self.model.state_dict(),
                        ckpt_path,
                        _use_new_zipfile_serialization=False
                    )

                    print(
                        f"    ✓ Best saved → {ckpt_path.name}"
                    )

                except Exception as e:

                    print(
                        f"    ⚠ Save failed: {e}"
                    )

            else:

                self.patience_count += 1

                if self.patience_count >= PATIENCE:

                    print(
                        f'  Early stop at epoch '
                        f'{epoch} '
                        f'(best epoch {self.best_ep})'
                    )

                    break

        # ===== LOAD BEST =====

        best_ckpt = MODEL_DIR / f"spec_fold{self.fold}.pt"

        if best_ckpt.exists():

            try:

                self.model.load_state_dict(
                    torch.load(
                        best_ckpt,
                        map_location=DEVICE
                    )
                )

                print(
                    f"Loaded best checkpoint "
                    f"(epoch {self.best_ep})"
                )

            except Exception as e:

                print(
                    f"WARNING: checkpoint load failed: {e}"
                )

        else:

            print(
                "WARNING: best checkpoint not found"
            )

        return history


print('Trainer defined ✓')

Trainer defined ✓


---
## Cell 7 — Smoke Test
Runs a tiny sanity check (SMOKE_N samples, 2 epochs) before full training.

In [20]:
def run_smoke_test(n: int = SMOKE_N, epochs: int = 2) -> bool:
    print(f'Smoke test: {n} samples, {epochs} epochs ...')
    smoke_df = eeg_high.head(n).copy()

    # Small split: 80 / 20
    split = int(0.8 * n)
    train_ds = HMSSpectrogramDataset(smoke_df.iloc[:split], TRAIN_EEG, train=True,  cache=False)
    val_ds   = HMSSpectrogramDataset(smoke_df.iloc[split:], TRAIN_EEG, train=False, cache=False)

    model   = build_model()
    trainer = SpectrogramTrainer(model, fold=0)
    trainer.train(train_ds, val_ds, max_epochs=epochs)

    del model, trainer, train_ds, val_ds
    torch.cuda.empty_cache(); gc.collect()
    print('Smoke test passed ✓')
    return True


run_smoke_test()

Smoke test: 50 samples, 2 epochs ...
Saving models to: /kaggle/working/models


train:   0%|          | 0/2 [00:00<?, ?it/s]

val:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch  1 | tr_kl=1.3019 | val_kl=1.2695 | val_auc=0.3973
    ✓ Best saved → spec_fold0.pt


train:   0%|          | 0/2 [00:00<?, ?it/s]

val:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch  2 | tr_kl=1.2063 | val_kl=1.2884 | val_auc=0.4125
Loaded best checkpoint (epoch 1)
Smoke test passed ✓


True

In [21]:
import shutil, os

shutil.rmtree(MODEL_DIR, ignore_errors=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("MODEL_DIR reset")
print(MODEL_DIR)

MODEL_DIR reset
/kaggle/working/models


---
## Cell 8 — GroupKFold Training (Patient-Level Split)

Identical split strategy as V2 — `GroupKFold(n_splits=3)` grouped by `patient_id`.

In [23]:
import shutil

usage = shutil.disk_usage("/kaggle/working")

print(f"Total : {usage.total/1e9:.2f} GB")
print(f"Used  : {usage.used/1e9:.2f} GB")
print(f"Free  : {usage.free/1e9:.2f} GB")

Total : 20.96 GB
Used  : 20.94 GB
Free  : 0.00 GB


In [25]:
from pathlib import Path
import os

def get_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except:
                pass
    return total

for p in Path("/kaggle/working").iterdir():
    try:
        size_gb = get_size(p) / 1e9
        print(f"{size_gb:.2f} GB   {p}")
    except Exception as e:
        print(p, e)

0.00 GB   /kaggle/working/models
0.00 GB   /kaggle/working/.virtual_documents
20.92 GB   /kaggle/working/spec_cache


In [26]:
import shutil
import gc

shutil.rmtree("/kaggle/working/spec_cache", ignore_errors=True)

gc.collect()

print("spec_cache deleted")

spec_cache deleted


In [27]:
import shutil

usage = shutil.disk_usage("/kaggle/working")

print(f"Free : {usage.free/1e9:.2f} GB")

Free : 20.94 GB


In [28]:
fold_results = {}   # fold_idx → {history, val_preds, val_targets, val_pids}

gkf    = GroupKFold(n_splits=N_FOLDS)
groups = eeg_high['patient_id'].values

for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(eeg_high, groups=groups)):
    print(f'\n{'='*60}')
    print(f'  FOLD {fold_idx+1} / {N_FOLDS}')
    print(f'  Train: {len(tr_idx):,} | Val: {len(val_idx):,}')
    print(f'  Val patients: {eeg_high.iloc[val_idx]["patient_id"].nunique():,}')
    print(f'{'='*60}')

    train_df = eeg_high.iloc[tr_idx].reset_index(drop=True)
    val_df   = eeg_high.iloc[val_idx].reset_index(drop=True)

    train_ds = HMSSpectrogramDataset(train_df, TRAIN_EEG, train=True)
    val_ds   = HMSSpectrogramDataset(val_df,   TRAIN_EEG, train=False)

    model   = build_model()
    trainer = SpectrogramTrainer(model, fold=fold_idx+1)
    history = trainer.train(train_ds, val_ds)

    # ── Collect val predictions ──────────────────────────────────────────────
    print('  Collecting val predictions ...')
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
        num_workers=N_WORKERS, pin_memory=True,
    )
    model.eval()
    all_preds, all_targets, all_pids = [], [], []
    with torch.no_grad():
        for specs, labels, pids in tqdm(val_loader, desc='predict', leave=False):
            logits = model(specs.to(DEVICE))
            probs  = F.softmax(logits, dim=-1).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(labels.numpy())
            all_pids.extend(pids.numpy().tolist())

    fold_results[fold_idx] = dict(
        history    = history,
        val_preds  = np.vstack(all_preds),
        val_targets= np.vstack(all_targets),
        val_pids   = np.array(all_pids),
    )

    # Save training curve
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_kl'], label='train'); axes[0].plot(history['val_kl'], label='val')
    axes[0].set_title(f'Fold {fold_idx+1} — KL Divergence'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['val_auc'], color='green')
    axes[1].set_title(f'Fold {fold_idx+1} — Val Macro AUC'); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'spec_training_curves_fold{fold_idx+1}.png', dpi=100)
    plt.close()

    del model, trainer, train_ds, val_ds, val_loader
    torch.cuda.empty_cache(); gc.collect()

print('\nAll folds complete ✓')


  FOLD 1 / 3
  Train: 7,264 | Val: 3,633
  Val patients: 629
Saving models to: /kaggle/working/models


train:   0%|          | 0/454 [00:00<?, ?it/s]

val:   0%|          | 0/114 [00:00<?, ?it/s]

  Epoch  1 | tr_kl=0.9592 | val_kl=0.8774 | val_auc=0.8331
    ✓ Best saved → spec_fold1.pt


train:   0%|          | 0/454 [00:00<?, ?it/s]

...

train:   0%|          | 0/454 [00:00<?, ?it/s]

val:   0%|          | 0/114 [00:00<?, ?it/s]

  Epoch  8 | tr_kl=0.4660 | val_kl=0.9241 | val_auc=0.8299
  Early stop at epoch 8 (best epoch 2)
Loaded best checkpoint (epoch 2)


predict:   0%|          | 0/114 [00:00<?, ?it/s]


All folds complete ✓


In [29]:
import zipfile
from pathlib import Path

zip_path = "/kaggle/working/HMS_SPECTROGRAM_BRANCH.zip"

files = [
    "/kaggle/working/models/spec_fold1.pt",
    "/kaggle/working/models/spec_fold2.pt",
    "/kaggle/working/models/spec_fold3.pt",
    "/kaggle/working/spec_training_curves_fold1.png",
    "/kaggle/working/spec_training_curves_fold2.png",
    "/kaggle/working/spec_training_curves_fold3.png",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p = Path(f)
        if p.exists():
            z.write(p, arcname=p.name)
            print(f"Added: {p.name}")
        else:
            print(f"Missing: {p}")

print("\nZIP created:")
print(zip_path)

Added: spec_fold1.pt
Added: spec_fold2.pt
Added: spec_fold3.pt
Added: spec_training_curves_fold1.png
Added: spec_training_curves_fold2.png
Added: spec_training_curves_fold3.png

ZIP created:
/kaggle/working/HMS_SPECTROGRAM_BRANCH.zip


---
## Cell 9 — Evaluation

Computes:
- **Fold-level**: KL Divergence, Accuracy, Macro AUC
- **Clinical per-class**: Sensitivity (Recall), Precision, F1, AUC
- **Confusion matrices** saved per fold

In [30]:
# ─── Helpers ──────────────────────────────────────────────────────────────────

def compute_kl(preds: np.ndarray, targets: np.ndarray, eps: float = 1e-7) -> float:
    p = np.clip(targets, eps, 1)
    q = np.clip(preds,   eps, 1)
    return float(np.mean(np.sum(p * np.log(p / q), axis=1)))


def compute_clinical_metrics(preds: np.ndarray, targets: np.ndarray) -> pd.DataFrame:
    """Per-class sensitivity, precision, recall, F1, AUC."""
    hard_preds   = preds.argmax(axis=1)
    hard_targets = targets.argmax(axis=1)
    rows = []
    for ci, cls in enumerate(CLASSES):
        tp  = ((hard_preds == ci) & (hard_targets == ci)).sum()
        fp  = ((hard_preds == ci) & (hard_targets != ci)).sum()
        fn  = ((hard_preds != ci) & (hard_targets == ci)).sum()
        tn  = ((hard_preds != ci) & (hard_targets != ci)).sum()
        sens = tp / (tp + fn + 1e-9)
        prec = tp / (tp + fp + 1e-9)
        f1   = 2*sens*prec / (sens + prec + 1e-9)
        try:
            binary_targets = (hard_targets == ci).astype(int)
            auc = roc_auc_score(binary_targets, preds[:, ci])
        except Exception:
            auc = 0.5
        rows.append(dict(class_=cls, sensitivity=sens, precision=prec, recall=sens, f1=f1, auc=auc))
    return pd.DataFrame(rows)


# ─── Per-fold evaluation ──────────────────────────────────────────────────────

fold_metrics_rows = []
all_preds_concat, all_targets_concat = [], []

for fi, res in fold_results.items():
    preds   = res['val_preds']
    targets = res['val_targets']
    hard_p  = preds.argmax(axis=1)
    hard_t  = targets.argmax(axis=1)

    kl_val  = compute_kl(preds, targets)
    acc     = (hard_p == hard_t).mean()
    try:
        auc = roc_auc_score(hard_t, preds, multi_class='ovr', average='macro')
    except:
        auc = 0.0

    fold_metrics_rows.append({'fold': fi+1, 'kl': kl_val, 'accuracy': acc, 'macro_auc': auc})
    print(f'Fold {fi+1} — KL: {kl_val:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f}')

    # Confusion matrix
    cm = confusion_matrix(hard_t, hard_p, labels=list(range(N_CLASSES)))
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Fold {fi+1} Confusion Matrix — V3 Spectrogram')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'spec_fold{fi+1}_confusion.png', dpi=100)
    plt.close()
    print(f'  Saved spec_fold{fi+1}_confusion.png')

    all_preds_concat.append(preds)
    all_targets_concat.append(targets)

fold_metrics_df = pd.DataFrame(fold_metrics_rows)
print('\n── Fold Summary ──')
print(fold_metrics_df.to_string(index=False))

# ── Aggregate clinical metrics across folds ──────────────────────────────────
all_preds   = np.vstack(all_preds_concat)
all_targets = np.vstack(all_targets_concat)

clinical_df = compute_clinical_metrics(all_preds, all_targets)
print('\n── Clinical Metrics (All Folds Pooled) ──')
print(clinical_df.to_string(index=False))

# Mean metrics
mean_kl  = fold_metrics_df['kl'].mean()
mean_auc = fold_metrics_df['macro_auc'].mean()
mean_acc = fold_metrics_df['accuracy'].mean()
seizure_sens = clinical_df.loc[clinical_df['class_']=='seizure', 'sensitivity'].values[0]
lrda_sens    = clinical_df.loc[clinical_df['class_']=='lrda',    'sensitivity'].values[0]
print(f'\nMean KL  : {mean_kl:.4f}')
print(f'Mean AUC : {mean_auc:.4f}')
print(f'Mean Acc : {mean_acc:.4f}')
print(f'Seizure Sensitivity : {seizure_sens:.4f} ({seizure_sens*100:.1f}%)')
print(f'LRDA Sensitivity    : {lrda_sens:.4f} ({lrda_sens*100:.1f}%)')

# Save metrics
fold_metrics_df.to_csv(OUT_DIR / 'spectrogram_metrics.csv', index=False)
clinical_df.to_csv(OUT_DIR / 'spec_clinical_metrics.csv', index=False)

# Save predictions
pred_rows = []
for fi, res in fold_results.items():
    df_p = pd.DataFrame(res['val_preds'], columns=[f'pred_{c}' for c in CLASSES])
    df_t = pd.DataFrame(res['val_targets'], columns=[f'true_{c}' for c in CLASSES])
    df_p['patient_id'] = res['val_pids']
    df_p['fold']       = fi + 1
    df_p['pred_class'] = res['val_preds'].argmax(axis=1)
    df_p['true_class'] = res['val_targets'].argmax(axis=1)
    pred_rows.append(pd.concat([df_p, df_t], axis=1))
pred_df = pd.concat(pred_rows, ignore_index=True)
pred_df.to_csv(OUT_DIR / 'spectrogram_predictions.csv', index=False)

print('\nSaved: spectrogram_metrics.csv, spec_clinical_metrics.csv, spectrogram_predictions.csv')
print('Evaluation complete ✓')

Fold 1 — KL: 0.8455 | Acc: 0.5299 | AUC: 0.8352
  Saved spec_fold1_confusion.png
Fold 2 — KL: 0.8206 | Acc: 0.5553 | AUC: 0.8435
  Saved spec_fold2_confusion.png
Fold 3 — KL: 0.8030 | Acc: 0.5545 | AUC: 0.8432
  Saved spec_fold3_confusion.png

── Fold Summary ──
 fold       kl  accuracy  macro_auc
    1 0.845534  0.529865   0.835237
    2 0.820585  0.555341   0.843494
    3 0.803033  0.554515   0.843240

── Clinical Metrics (All Folds Pooled) ──
 class_  sensitivity  precision   recall       f1      auc
seizure     0.605699   0.623758 0.605699 0.614595 0.856947
    lpd     0.466899   0.589011 0.466899 0.520894 0.848282
    gpd     0.631579   0.566775 0.631579 0.597425 0.912308
   lrda     0.330656   0.222122 0.330656 0.265734 0.777262
   grda     0.494565   0.493416 0.494565 0.493990 0.867782
  other     0.583635   0.595037 0.583635 0.589281 0.767610

Mean KL  : 0.8231
Mean AUC : 0.8407
Mean Acc : 0.5466
Seizure Sensitivity : 0.6057 (60.6%)
LRDA Sensitivity    : 0.3307 (33.1%)

Saved: 

In [34]:
model = build_model()

print(model)

SpectrogramEfficientNet(
  (backbone): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
        

---
## Cell 10 — Grad-CAM Explainability

Grad-CAM visualises which **time-frequency regions** drive each prediction.
We hook the last convolutional block of EfficientNet-B0 (`backbone.features[-1]`).

In [37]:
class GradCAM:
    """
    Grad-CAM for EfficientNet-B0 16-channel spectrogram model.
    Targets the last feature block.
    """

    def __init__(self, model: SpectrogramEfficientNet):
        self.model     = model
        self.gradients = None
        self.features  = None
        self._register_hooks()

    def _register_hooks(self):
        # Last feature block
        target_layer = self.model.backbone.features[-1]

        def fwd_hook(module, inp, out):
            self.features = out

        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self._fwd_handle = target_layer.register_forward_hook(fwd_hook)
        self._bwd_handle = target_layer.register_full_backward_hook(bwd_hook)

    def remove_hooks(self):
        self._fwd_handle.remove()
        self._bwd_handle.remove()

    def __call__(
        self,
        spec: torch.Tensor,   # (16, H, W) — unbatched
        target_class: Optional[int] = None,
    ) -> Tuple[np.ndarray, int, float]:
        """
        Returns:
            cam      : (H, W) heat-map in [0,1]
            pred_cls : predicted class index
            conf     : softmax confidence
        """
        self.model.eval()
        inp = spec.unsqueeze(0).to(DEVICE).requires_grad_(False)
        inp.requires_grad_(True)

        logits = self.model(inp)
        probs  = F.softmax(logits, dim=-1)
        if target_class is None:
            target_class = probs.argmax(dim=-1).item()
        conf = probs[0, target_class].item()

        self.model.zero_grad()
        logits[0, target_class].backward()

        grads    = self.gradients[0]              # (C, h, w)
        feats    = self.features[0]               # (C, h, w)
        weights  = grads.mean(dim=(1, 2))         # (C,)
        cam      = (weights[:, None, None] * feats).sum(0)  # (h, w)
        cam      = F.relu(cam)
        cam      = cam - cam.min()
        cam_max  = cam.max()
        if cam_max > 1e-8:
            cam = cam / cam_max

        # Resize to input spatial size
        cam_up = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=(SPEC_H, SPEC_W),
            mode='bilinear', align_corners=False
        ).squeeze().detach().cpu().numpy()

        return cam_up, int(target_class), conf


def generate_gradcam_examples(
    model_path: Path,
    n_examples: int = 6,
    n_examples_per_class: int = 1,
) -> None:
    """Load best model, run Grad-CAM on representative samples, save figure."""

    # Load saved fold-1 model
    model = build_model()
    if model_path.exists():
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    grad_cam = GradCAM(model)

    # Pick one confirmed sample per class
    samples = []
    for cls in CLASSES:
        cls_rows = eeg_high[eeg_high['expert_consensus'] == cls].head(1)
        if len(cls_rows):
            samples.append(cls_rows.iloc[0])
    samples = samples[:n_examples]

    if len(samples) == 0:
        print("No Grad-CAM examples found — skipping visualization.")
        return
    
    fig, axes = plt.subplots(
        len(samples),
        3,
        figsize=(14, 4*len(samples))
    )
    if len(samples) == 1:
        axes = [axes]

    for i, row in enumerate(samples):
        eeg_id = int(row['eeg_id'])
        parquet = TRAIN_EEG / f'{eeg_id}.parquet'
        if not parquet.exists():
            continue
        df_eeg   = pd.read_parquet(parquet)
        bipolar  = eeg_to_bipolar(df_eeg)
        spec_t   = build_spectrogram_tensor(bipolar)

        cam, pred_cls, conf = grad_cam(spec_t)

        # Show mean spectrogram across channels
        mean_spec = spec_t.mean(0).numpy()
        true_cls  = row['expert_consensus']

        axes[i][0].imshow(mean_spec, origin='lower', aspect='auto',
                           cmap='inferno', vmin=0, vmax=1)
        axes[i][0].set_title(f'Mean Mel Spec | True: {true_cls}', fontsize=9)
        axes[i][0].axis('off')

        axes[i][1].imshow(cam, origin='lower', aspect='auto',
                           cmap='jet', vmin=0, vmax=1)
        axes[i][1].set_title(f'Grad-CAM', fontsize=9)
        axes[i][1].axis('off')

        # Overlay
        axes[i][2].imshow(mean_spec, origin='lower', aspect='auto',
                           cmap='inferno', vmin=0, vmax=1)
        axes[i][2].imshow(cam, origin='lower', aspect='auto',
                           cmap='jet', alpha=0.45, vmin=0, vmax=1)
        axes[i][2].set_title(
            f'Overlay | Pred: {CLASSES[pred_cls]} ({conf*100:.1f}%)', fontsize=9
        )
        axes[i][2].axis('off')

    plt.suptitle('Grad-CAM: EEG Spectrogram Explainability (V3)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'gradcam_examples.png', dpi=100, bbox_inches='tight')
    plt.close()
    grad_cam.remove_hooks()
    del model
    torch.cuda.empty_cache()
    print('Saved gradcam_examples.png ✓')


print('Grad-CAM class defined ✓')

Grad-CAM class defined ✓


---
## Cell 11 — Full Evaluation Dashboard

In [38]:
def save_evaluation_dashboard() -> None:
    """Save comprehensive evaluation figure: class metrics + ROC curves."""
    preds   = all_preds
    targets = all_targets
    hard_p  = preds.argmax(axis=1)
    hard_t  = targets.argmax(axis=1)

    fig = plt.figure(figsize=(18, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig)

    # ── 1. Per-class Bar chart ────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    x = np.arange(N_CLASSES)
    width = 0.25
    ax1.bar(x - width, clinical_df['sensitivity'], width, label='Sensitivity')
    ax1.bar(x,         clinical_df['precision'],   width, label='Precision')
    ax1.bar(x + width, clinical_df['f1'],          width, label='F1')
    ax1.set_xticks(x); ax1.set_xticklabels(CLASSES, rotation=30, ha='right')
    ax1.set_ylim(0, 1.05); ax1.legend(); ax1.grid(axis='y')
    ax1.set_title('Per-Class Clinical Metrics')

    # ── 2. Per-class AUC ─────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.barh(clinical_df['class_'], clinical_df['auc'], color='steelblue')
    ax2.set_xlim(0, 1); ax2.axvline(0.5, color='red', linestyle='--', lw=1)
    ax2.set_title('Per-Class AUC'); ax2.grid(axis='x')

    # ── 3. Confusion Matrix (all folds pooled) ────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    cm = confusion_matrix(hard_t, hard_p, labels=list(range(N_CLASSES)))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES,
                cmap='Blues', ax=ax3)
    ax3.set_xlabel('Predicted'); ax3.set_ylabel('True')
    ax3.set_title('Pooled Confusion Matrix')

    # ── 4. ROC Curves ────────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, :2])
    for ci, cls in enumerate(CLASSES):
        binary_t = (hard_t == ci).astype(int)
        fpr, tpr, _ = roc_curve(binary_t, preds[:, ci])
        roc_auc = sklearn_auc(fpr, tpr)
        ax4.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})')
    ax4.plot([0,1],[0,1], 'k--', lw=1)
    ax4.set_xlabel('FPR'); ax4.set_ylabel('TPR')
    ax4.set_title('Per-Class ROC Curves — V3 Spectrogram')
    ax4.legend(loc='lower right'); ax4.grid(True)

    # ── 5. Fold KL summary ────────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.bar([f'Fold {r["fold"]}' for _, r in fold_metrics_df.iterrows()],
            fold_metrics_df['kl'], color=['#3498db','#e67e22','#27ae60'])
    ax5.set_ylabel('KL Divergence'); ax5.set_title('Per-Fold KL')
    ax5.grid(axis='y')
    for _, r in fold_metrics_df.iterrows():
        ax5.text(r['fold']-1, r['kl']+0.002, f"{r['kl']:.4f}", ha='center', fontsize=9)

    plt.suptitle('HMS V3 — Spectrogram Branch Evaluation', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'spectrogram_evaluation.png', dpi=100, bbox_inches='tight')
    plt.close()
    print('Saved spectrogram_evaluation.png ✓')


save_evaluation_dashboard()

# Grad-CAM
best_model_path = MODEL_DIR / 'spec_fold1.pt'
generate_gradcam_examples(best_model_path)

# Save best model copy
import shutil
if best_model_path.exists():
    shutil.copy2(best_model_path, OUT_DIR / 'best_spectrogram_model.pt')
    print('Saved best_spectrogram_model.pt ✓')

print('\nAll outputs saved ✓')

Saved spectrogram_evaluation.png ✓
No Grad-CAM examples found — skipping visualization.
Saved best_spectrogram_model.pt ✓

All outputs saved ✓


---
## Cell 12 — V2 vs V3 Comparison Report

Automatically prints the comparison table and written analysis.

> **V2 baselines** are hard-coded from the known V2 results.
> After running V3, replace `V3_*` values with actual outputs from Cell 9.

In [39]:
# ── V2 baselines (from completed V2 run) ──────────────────────────────────────
V2_KL               = 0.84
V2_AUC              = 0.85
V2_ACC              = None    # not reported in V2 summary
V2_SEIZURE_SENS     = 0.39
V2_LRDA_SENS        = 0.26

# ── V3 results (from this run) ────────────────────────────────────────────────
V3_KL           = mean_kl
V3_AUC          = mean_auc
V3_ACC          = mean_acc
V3_SEIZURE_SENS = seizure_sens
V3_LRDA_SENS    = lrda_sens

# ── Print comparison table ────────────────────────────────────────────────────
print('\n' + '='*65)
print(' V2 EEG-only  vs  V3 Spectrogram — Final Comparison')
print('='*65)
header = f"{'Metric':<28} {'EEG Branch (V2)':>16} {'Spectrogram (V3)':>18}"
print(header)
print('-'*65)

def fmt(v, fmt_str='%.4f'):
    return fmt_str % v if v is not None else 'N/A'

rows = [
    ('KL Divergence (↓)',       V2_KL,           V3_KL),
    ('Macro AUC (↑)',           V2_AUC,          V3_AUC),
    ('Accuracy (↑)',            V2_ACC,          V3_ACC),
    ('Seizure Sensitivity (↑)', V2_SEIZURE_SENS, V3_SEIZURE_SENS),
    ('LRDA Sensitivity (↑)',    V2_LRDA_SENS,    V3_LRDA_SENS),
]
for name, v2, v3 in rows:
    delta = ''
    if v2 is not None and v3 is not None:
        diff = v3 - v2
        arrow = '▲' if diff > 0 else '▼'
        delta = f'  {arrow}{abs(diff):.4f}'
        if name.startswith('KL'):      # lower is better
            arrow = '▲' if diff < 0 else '▼'
            delta = f'  {arrow}{abs(diff):.4f}'
    print(f"{name:<28} {fmt(v2):>16} {fmt(v3):>18}{delta}")
print('='*65)

# ── Written analysis ──────────────────────────────────────────────────────────
print('\n── Written Analysis ────────────────────────────────────────────────')

seizure_improved = V3_SEIZURE_SENS > V2_SEIZURE_SENS
lrda_improved    = V3_LRDA_SENS    > V2_LRDA_SENS
auc_improved     = V3_AUC          > V2_AUC
kl_improved      = V3_KL           < V2_KL

print(f"""
1. Did spectrograms improve seizure detection?
   {'YES' if seizure_improved else 'NO'} — Seizure sensitivity changed from {V2_SEIZURE_SENS*100:.1f}%
   to {V3_SEIZURE_SENS*100:.1f}% ({'+' if seizure_improved else ''}{(V3_SEIZURE_SENS-V2_SEIZURE_SENS)*100:.1f} pp).
   {'Mel spectrograms capture the broadband high-frequency bursts characteristic of'
    ' seizures, which may be easier to discriminate in time-frequency space than in'
    ' raw waveform representations.' if seizure_improved else
    'The CNN-BiLSTM model may already extract sufficient temporal dynamics for seizure'
    ' detection. Spectrograms may benefit from additional pre-training or larger models.'}

2. Did spectrograms improve LRDA detection?
   {'YES' if lrda_improved else 'NO'} — LRDA sensitivity changed from {V2_LRDA_SENS*100:.1f}%
   to {V3_LRDA_SENS*100:.1f}% ({'+' if lrda_improved else ''}{(V3_LRDA_SENS-V2_LRDA_SENS)*100:.1f} pp).
   {'Lateralised rhythmic delta activity exhibits a distinctive spectral signature in'
    ' the 1-3 Hz band. Mel spectrograms with low-frequency emphasis effectively'
    ' capture this pattern.' if lrda_improved else
    'LRDA patterns are rhythmic and low-frequency; the BiLSTM in V2 may capture'
    ' their temporal periodicity more directly than a static spectrogram image.'}

3. Which modality performs better overall?
   {'Spectrogram (V3)' if (auc_improved or kl_improved) else 'EEG (V2)'} achieves
   better overall performance:
   - KL:  V2={V2_KL:.4f} vs V3={V3_KL:.4f}  ({'V3 better ↓' if kl_improved else 'V2 better'})
   - AUC: V2={V2_AUC:.4f} vs V3={V3_AUC:.4f}  ({'V3 better ↑' if auc_improved else 'V2 better'})

4. What complementary information do spectrograms capture?
   Spectrograms encode time-frequency structure explicitly, making it easier for
   convolutional networks to detect:
   - Frequency-specific power changes (e.g., delta bursts in LRDA/GRDA)
   - Spectral shape of ictal discharges (seizure)
   - Rhythmicity through repeating time-frequency patterns
   The EEG branch, by contrast, preserves fine temporal dynamics and phase
   relationships not visible in power spectrograms.

5. Is multimodal fusion justified?
   {'YES' if (seizure_improved or lrda_improved or auc_improved) else 'PARTIALLY'} —
   The two modalities capture complementary features:
   - EEG branch: temporal dynamics, waveform morphology, inter-channel phase
   - Spectrogram branch: frequency-domain patterns, rhythmicity, spectral shape
   A fusion model (V4) is scientifically motivated. Expected fusion gains:
   - Improved minority class sensitivity (seizure + LRDA)
   - Reduced KL divergence
   - More robust predictions under artefact conditions
""")

# ── Save summary CSV ──────────────────────────────────────────────────────────
comparison_df = pd.DataFrame([
    {'metric': 'KL_divergence',       'v2_eeg': V2_KL,           'v3_spectrogram': V3_KL},
    {'metric': 'macro_AUC',           'v2_eeg': V2_AUC,          'v3_spectrogram': V3_AUC},
    {'metric': 'accuracy',            'v2_eeg': V2_ACC,          'v3_spectrogram': V3_ACC},
    {'metric': 'seizure_sensitivity', 'v2_eeg': V2_SEIZURE_SENS, 'v3_spectrogram': V3_SEIZURE_SENS},
    {'metric': 'lrda_sensitivity',    'v2_eeg': V2_LRDA_SENS,    'v3_spectrogram': V3_LRDA_SENS},
])
comparison_df.to_csv(OUT_DIR / 'v2_v3_comparison.csv', index=False)
print('Saved v2_v3_comparison.csv ✓')

print('\n' + '='*65)
print(' OUTPUT FILES SUMMARY')
print('='*65)
outputs = [
    'best_spectrogram_model.pt',
    'spectrogram_metrics.csv',
    'spectrogram_predictions.csv',
    'spec_clinical_metrics.csv',
    'spectrogram_evaluation.png',
    'spec_fold1_confusion.png',
    'spec_fold2_confusion.png',
    'spec_fold3_confusion.png',
    'spec_training_curves_fold1.png',
    'spec_training_curves_fold2.png',
    'spec_training_curves_fold3.png',
    'gradcam_examples.png',
    'v2_v3_comparison.csv',
]
for f in outputs:
    p = OUT_DIR / f
    size = f'{p.stat().st_size/1024:.0f} KB' if p.exists() else 'NOT FOUND'
    mark = '✓' if p.exists() else '✗'
    print(f'  {mark} {f:<45} {size}')

print('\n🏁 Version 3 Spectrogram Benchmark complete.')


 V2 EEG-only  vs  V3 Spectrogram — Final Comparison
Metric                        EEG Branch (V2)   Spectrogram (V3)
-----------------------------------------------------------------
KL Divergence (↓)                      0.8400             0.8231  ▲0.0169
Macro AUC (↑)                          0.8500             0.8407  ▼0.0093
Accuracy (↑)                              N/A             0.5466
Seizure Sensitivity (↑)                0.3900             0.6057  ▲0.2157
LRDA Sensitivity (↑)                   0.2600             0.3307  ▲0.0707

── Written Analysis ────────────────────────────────────────────────

1. Did spectrograms improve seizure detection?
   YES — Seizure sensitivity changed from 39.0%
   to 60.6% (+21.6 pp).
   Mel spectrograms capture the broadband high-frequency bursts characteristic of seizures, which may be easier to discriminate in time-frequency space than in raw waveform representations.

2. Did spectrograms improve LRDA detection?
   YES — LRDA sensitivity chan

In [40]:
import zipfile
from pathlib import Path

zip_path = "/kaggle/working/HMS_V3_SPECTROGRAM_RESULTS.zip"

files_to_zip = [
    "/kaggle/working/best_spectrogram_model.pt",

    "/kaggle/working/spectrogram_metrics.csv",
    "/kaggle/working/spectrogram_predictions.csv",
    "/kaggle/working/spec_clinical_metrics.csv",
    "/kaggle/working/v2_v3_comparison.csv",

    "/kaggle/working/spectrogram_evaluation.png",

    "/kaggle/working/spec_fold1_confusion.png",
    "/kaggle/working/spec_fold2_confusion.png",
    "/kaggle/working/spec_fold3_confusion.png",

    "/kaggle/working/spec_training_curves_fold1.png",
    "/kaggle/working/spec_training_curves_fold2.png",
    "/kaggle/working/spec_training_curves_fold3.png",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:

    for f in files_to_zip:

        p = Path(f)

        if p.exists():
            z.write(p, arcname=p.name)
            print(f"✓ Added: {p.name}")
        else:
            print(f"✗ Missing: {p.name}")

print("\n" + "="*60)
print("ZIP CREATED")
print("="*60)
print(zip_path)

✓ Added: best_spectrogram_model.pt
✓ Added: spectrogram_metrics.csv
✓ Added: spectrogram_predictions.csv
✓ Added: spec_clinical_metrics.csv
✓ Added: v2_v3_comparison.csv
✓ Added: spectrogram_evaluation.png
✓ Added: spec_fold1_confusion.png
✓ Added: spec_fold2_confusion.png
✓ Added: spec_fold3_confusion.png
✓ Added: spec_training_curves_fold1.png
✓ Added: spec_training_curves_fold2.png
✓ Added: spec_training_curves_fold3.png

ZIP CREATED
/kaggle/working/HMS_V3_SPECTROGRAM_RESULTS.zip
